# Lecture 14.3 YOLO

**Topic:** the YOLO family, why one-stage detection changed practical computer vision, and how to train and run a modern YOLO pipeline  
**Audience:** students who already know the general idea of detection and want a deployment-oriented view  
**Goal:** understand the original YOLO intuition, the evolution to modern YOLO systems, and the practical Ultralytics workflow

This version expands the theory substantially and connects the historical idea to the toolchain students are most likely to meet in real projects.


## Outline

1. Why YOLO mattered historically  
2. The original grid-based idea  
3. How modern YOLO models differ from YOLOv1  
4. Training signals: box loss, objectness, classes  
5. Real training and inference with Ultralytics  
6. Applications, tradeoffs, and exercises


In [ ]:
from __future__ import annotations

import math
import random
from pathlib import Path

import numpy as np

MPL_AVAILABLE = True
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
except Exception as exc:
    MPL_AVAILABLE = False
    print(f"matplotlib is not available: {exc}")

PIL_AVAILABLE = True
try:
    from PIL import Image
except Exception as exc:
    PIL_AVAILABLE = False
    print(f"Pillow is not available: {exc}")

TORCH_AVAILABLE = True
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
except Exception as exc:
    TORCH_AVAILABLE = False
    print(f"PyTorch is not available: {exc}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)

print(f"matplotlib available: {MPL_AVAILABLE}")
print(f"Pillow available: {PIL_AVAILABLE}")
print(f"PyTorch available: {TORCH_AVAILABLE}")


## 1. Why YOLO was such a big moment

Before YOLO, many influential detectors followed a slower pipeline:

- generate region proposals  
- classify regions  
- refine boxes

YOLO changed the framing by treating detection as a **single end-to-end prediction problem**. Instead of classifying many crops independently, the network looked at the entire image once and predicted all objects in one pass.

This mattered for two reasons:

- it was faster  
- it changed how people mentally organized detection

The historical importance of YOLO is not only that it was quick. It made one-stage detection feel like the natural way to think about real-time perception.


![YOLO grid intuition](images/yolo_grid.svg)

The original intuition is visible in the diagram: image space is organized into prediction locations, and each location is responsible for making structured predictions. Even though modern YOLO implementations are much more sophisticated than YOLOv1, this global one-pass idea is still the core identity of the family.


## 2. The original YOLO idea in plain language

YOLOv1 divided the image into a grid. Each grid cell predicted:

- whether an object center was inside the cell  
- one or more bounding boxes  
- class probabilities

What was powerful about this?

The model reasoned about the whole scene at once. A region was not classified in isolation. The network learned global context and local localization together.

For teaching, the most important conceptual takeaway is:

**YOLO turns detection into one coherent prediction problem rather than a proposal pipeline.**


## 3. YOLO then and YOLO now

Students should not confuse “the original YOLO paper” with “everything people now call YOLO.”

### YOLOv1

- highly influential  
- grid-based formulation  
- simple compared with later versions  
- weaker on small objects

### Later generations

Modern YOLO systems improved the original idea with:

- stronger backbones  
- feature pyramids and better multi-scale heads  
- improved label assignment  
- stronger data augmentation  
- better losses and training heuristics  
- better deployment tooling

So when engineers say “YOLO” today, they usually mean a practical one-stage detection family, not only the exact 2015 architecture.


![Modern YOLO workflow](images/yolo_pipeline.svg)

This modern workflow is one reason YOLO stayed relevant. It is not only a paper architecture now. It is a full training and deployment ecosystem.


## 4. What a YOLO model predicts

A modern YOLO head usually predicts, at many candidate locations and scales:

- box geometry  
- objectness  
- class scores

Conceptually, training combines several objectives:

$$
L = L_{box} + L_{obj} + L_{cls}
$$

Where:

- $L_{box}$ improves localization  
- $L_{obj}$ improves objectness confidence  
- $L_{cls}$ improves classification

You do not need to memorize every coefficient to use YOLO well. What matters is understanding that YOLO must learn where the box is, whether the object exists, and what class it belongs to.


## 4.1 YOLOv1 in much more detail

The original YOLO paper proposed a very specific formulation:

- divide the image into an `S x S` grid
- let each cell predict a fixed number of boxes
- let each cell predict class probabilities

One subtle point is responsibility assignment. A cell was considered responsible for an object if the object center fell inside that cell.

This is conceptually elegant, but it also explains some YOLOv1 limitations:

- a single cell can become overloaded if objects are close together
- small objects are hard because the grid is coarse
- localization flexibility is limited by the assignment scheme

This is why later YOLO models evolved away from the exact YOLOv1 formulation while keeping the overall one-stage spirit.


## 4.2 Backbone, neck, and head

Modern YOLO systems are easiest to understand if students separate the architecture into three parts.

### Backbone

The backbone extracts visual features from the image. It plays the same broad role as the feature extractor in many CNN systems.

### Neck

The neck mixes information across scales. In modern detectors this is critical because detection must work for:

- small nearby details
- medium objects
- large dominant objects

Feature pyramid style mixing helps the detector use both shallow high-resolution information and deep semantic information.

### Head

The head outputs the final structured predictions: box geometry, objectness, and class scores.

Thinking in terms of backbone / neck / head is one of the best ways for students to read modern detector diagrams without getting lost.


## 4.3 Label assignment and why it matters

A detector does not learn from boxes magically. The training pipeline must decide which predictions are treated as positive examples for which ground-truth objects.

This is called **label assignment**.

Why is it so important?

Because object detection is not like classification, where each image has one direct label. In detection there may be many candidate predictions per image, and the training process must decide:

- which predictions should move toward a given object
- which predictions should be treated as background

Modern YOLO systems improved dramatically over time partly because this assignment logic became smarter.

Students do not need to memorize every assignment algorithm, but they should understand that:

**detection quality depends not only on the network, but also on how ground truth is matched to predictions during training.**


## 4.4 Box regression loss intuition

When students first see YOLO losses, they often expect the detector to optimize only classification-like objectives. But a detector also has to optimize geometry.

Box regression losses try to make predicted boxes align well with ground-truth boxes. Older systems often used coordinate-wise losses. Modern systems frequently use IoU-based losses or their improved variants because they better reflect the geometric quality of the box.

Why is that important?

A coordinate error of 5 pixels can mean very different things for:

- a huge object
- a tiny object

IoU-based thinking is therefore often more meaningful than raw coordinate error.


## 4.5 Multi-scale prediction

One of the big reasons later YOLO versions became stronger is multi-scale prediction.

If the detector only predicts at one feature-map resolution, it struggles to handle the full range of object sizes. Modern YOLO systems therefore predict at several scales, often using a feature pyramid style design.

This helps because:

- shallow higher-resolution features help with small objects
- deeper semantic features help with larger and more contextual reasoning

For students, the key takeaway is simple:

**modern YOLO is not “one grid and done.” It is a multi-scale detector built to handle object size variation.**


## 5. Objectness, confidence, and final scores

One source of beginner confusion is the relationship between objectness and class probability.

Objectness answers:

- “is there an object here at all?”

Class probability answers:

- “if there is an object, which class is it?”

The displayed score often reflects a combination of both. This is why a box can look visually reasonable but still receive a modest final confidence if either objectness or class confidence is weak.


In [ ]:
# A tiny intuition demo: objectness and class score are different signals.

predictions = [
    {"box": [20, 25, 120, 160], "objectness": 0.95, "class_prob_person": 0.90},
    {"box": [18, 22, 118, 158], "objectness": 0.82, "class_prob_person": 0.91},
    {"box": [200, 80, 260, 130], "objectness": 0.60, "class_prob_person": 0.20},
]

for pred in predictions:
    score = pred["objectness"] * pred["class_prob_person"]
    print(pred["box"], "final person score =", round(score, 3))


In [ ]:
ULTRALYTICS_AVAILABLE = True
try:
    from ultralytics import YOLO
except Exception as exc:
    ULTRALYTICS_AVAILABLE = False
    print(f"Ultralytics is not available: {exc}")

print(f"Ultralytics available: {ULTRALYTICS_AVAILABLE}")


## 6. Why YOLO is popular in practice

YOLO became popular not only because it is fast, but because it gives a highly practical package:

- good speed  
- reasonable to strong accuracy  
- clear training APIs  
- export and deployment options  
- easy experimentation

That combination makes it a default choice in many student and production settings:

- CCTV analytics  
- robotics  
- industrial inspection  
- sports analytics  
- drone video  
- edge devices


In [ ]:
if not ULTRALYTICS_AVAILABLE:
    print("Install with: pip install ultralytics")
else:
    model = YOLO("yolov8n.pt")

    print("Model loaded.")
    print("Suggested quick classroom training command:")
    print("model.train(data='coco8.yaml', epochs=3, imgsz=640, batch=16)")


## 7. Why the notebook uses the Ultralytics package

For a teaching notebook, the `ultralytics` package is the shortest route from theory to a real working detector.

It gives students:

- pretrained checkpoints  
- training commands with real dataset configs  
- prediction and visualization utilities  
- export support

This matters because classroom time is limited. If the infrastructure is too heavy, students spend the whole lecture on setup instead of learning the concepts.


## 7.1 Why Ultralytics is pedagogically useful

Students often meet YOLO through the `ultralytics` package rather than by re-implementing the architecture from scratch. This is actually a good thing in a course setting.

Why?

Because it lets the lecture focus on the core learning goals:

- dataset format
- training loop logic
- validation behavior
- prediction inspection
- deployment workflow

If we forced students to hand-build everything before they could see a working detector, many would spend the whole lecture on infrastructure and miss the main conceptual story.


## 8. Real dataset example and why `coco8` is used

The notebook suggests `coco8.yaml`, a tiny real subset of COCO maintained for quick experiments.

Why use a small subset in a lecture?

- it is still a real dataset format  
- the API and workflow are realistic  
- training finishes quickly enough for demonstrations

The lesson is not “train a state-of-the-art detector in three minutes.” The lesson is:

**see the full real workflow without waiting all day.**


## 9. What to tell students about YOLO tradeoffs

YOLO is excellent when:

- speed matters  
- deployment matters  
- the dataset fits standard detection assumptions  
- you want a practical baseline quickly

YOLO is not automatically the best choice when:

- extreme localization precision matters more than speed  
- you need instance segmentation rather than boxes  
- another architecture fits the benchmark regime better

The right summary is not “YOLO wins everything.” The right summary is:

**YOLO is one of the strongest practical defaults.**


## 9.1 Where YOLO fits in a model selection conversation

Students often ask a version of the same question:

“If YOLO is so practical, why not use it for everything?”

This is a useful question because it teaches architectural judgment. YOLO is a great default when:

- latency matters
- the task is standard object detection
- deployment simplicity matters
- we want a strong baseline quickly

But model selection is always contextual. In some projects another detector or even another task formulation may be better:

- segmentation if boundaries matter
- tracking if temporal identity matters
- OCR pipelines if text extraction matters

So the real lesson is not “YOLO wins.” The real lesson is “YOLO is a strong answer to a specific family of practical constraints.”


In [ ]:
if not ULTRALYTICS_AVAILABLE:
    print("Skipping training example: Ultralytics not available.")
else:
    # Uncomment to run a real short training session.
    # results = model.train(
    #     data="coco8.yaml",
    #     epochs=3,
    #     imgsz=640,
    #     batch=16,
    #     project="runs/lecture14",
    #     name="yolo_coco8_demo",
    # )
    print("Training cell prepared. Remove comments to launch a real run.")


## 10. Common mistakes and failure modes

Common beginner mistakes:

- training with the wrong label format  
- comparing models trained at very different image sizes  
- confusing confidence threshold changes with actual model improvement  
- using too little data and drawing strong conclusions

Typical failure modes:

- weak performance on small objects  
- duplicate boxes if thresholds are not tuned  
- unstable results on rare classes  
- overconfidence on easy backgrounds

As with all detectors, qualitative inspection is essential. Students should check both:

- the number of detections  
- the quality of localization and class assignments


In [ ]:
if not ULTRALYTICS_AVAILABLE:
    print("Skipping inference example: Ultralytics not available.")
else:
    from ultralytics.utils import ASSETS

    sample_image = ASSETS / "bus.jpg"
    results = model.predict(sample_image, conf=0.25, verbose=False)
    first = results[0]

    print("Predicted classes:", first.boxes.cls.cpu().numpy()[:10])
    print("Predicted confidences:", np.round(first.boxes.conf.cpu().numpy()[:10], 3))

    if MPL_AVAILABLE:
        plotted = first.plot()
        plt.figure(figsize=(10, 6))
        plt.imshow(plotted[..., ::-1])
        plt.axis("off")
        plt.title("YOLO predictions on a real image")
        plt.show()


## 11. Deployment perspective

YOLO is especially strong when the lecture needs to connect deep learning theory to real deployment.

Why?

Because a typical workflow can cover:

- training  
- validation  
- inference  
- export to ONNX or other formats  
- edge or server deployment

This makes YOLO pedagogically valuable: students see not only an architecture, but a realistic engineering path from data to deployed detector.


## 11.1 The deployment story in more detail

YOLO became especially dominant in educational and industrial settings because it connects elegantly to deployment.

A realistic workflow might look like this:

1. annotate or prepare a dataset
2. fine-tune a pretrained checkpoint
3. validate with mAP and qualitative inspection
4. run batch or streaming inference
5. export to a runtime-friendly format
6. deploy on GPU, server, edge device, or embedded system

This matters pedagogically because students can see the whole machine learning lifecycle, not only the “training in a notebook” phase.


## Exercises

1. Train on `coco8.yaml` for longer and compare predictions.  
2. Replace `yolov8n.pt` with a larger checkpoint and compare speed/accuracy.  
3. Change confidence thresholds and inspect the precision-recall tradeoff qualitatively.  
4. Export the trained model and document the deployment workflow.


## References

- Expanded from the course materials and [deepmachinelearning.ru](https://deepmachinelearning.ru/docs/Neural-networks/Object-detection).  
- Original YOLO paper: *You Only Look Once: Unified, Real-Time Object Detection*.  
- Ultralytics documentation for modern training and deployment workflows.


In [ ]:
# Exercise scaffold: compare confidence thresholds

if not ULTRALYTICS_AVAILABLE:
    print("Ultralytics is required for the YOLO exercise.")
else:
    for conf in [0.15, 0.25, 0.50]:
        results = model.predict(sample_image, conf=conf, verbose=False)
        count = len(results[0].boxes)
        print(f"conf={conf:.2f} -> detections={count}")
